# 29 — Semisupervised Training

## Objetivo

Evaluar si el uso de datos no etiquetados mejora el baseline supervisado disponible para `H6_violation`.

La comparación es:

$$
M_{sup}(L)
\quad vs \quad
M_{ssl}(L \cup \hat{U})
$$

donde `\hat{U}` contiene únicamente pseudo-etiquetas de alta confianza.

Reglas metodológicas:

- reutilizar exactamente los folds del notebook 28;
- mantener el fold de test completamente fuera del pseudo-etiquetado;
- no convertir Y desconocida en 0;
- no usar procedencia, dominio ni fuente como predictores;
- registrar cada pseudo-etiqueta y ronda;
- no elegir todavía un modelo final;
- mantener H1/H4/H5 bloqueados mientras no exista soporte de evaluación suficiente.

Dado el cambio de dominio `WEB_DOM → ANDROID_UI`, la configuración primaria usa un filtro conservador de solapamiento de X.


## 1. Configuración

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

SEMISUP_INTERIM_DIR = (
    PROJECT_DIR / "data" / "semisupervised" / "interim"
)

SEMISUP_REPORTS_DIR = (
    PROJECT_DIR / "reports" / "tables" / "semisupervised"
)

SEMISUP_REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURES = [
    "structure_depth",
    "interactive_control_count",
    "interactive_label_ratio"
]

ALL_TARGETS = [
    "H1_violation",
    "H4_violation",
    "H5_violation",
    "H6_violation"
]

RANDOM_STATE = 42
PREDICTION_THRESHOLD = 0.50

print("PROJECT_DIR:", PROJECT_DIR)


PROJECT_DIR: /Users/uriel/Documents/machine_learning


## 2. Cargar master, folds y baseline

In [2]:
master_path = (
    SEMISUP_INTERIM_DIR
    / "semisupervised_proxy_master_dataset_v2.csv"
)

fold_assignment_path = (
    SEMISUP_REPORTS_DIR
    / "baseline_grouped_fold_assignments.csv"
)

baseline_oof_path = (
    SEMISUP_REPORTS_DIR
    / "baseline_oof_predictions.csv"
)

continuation_path = (
    SEMISUP_REPORTS_DIR
    / "baseline_continuation_decision.csv"
)

required_paths = [
    master_path,
    fold_assignment_path,
    baseline_oof_path,
    continuation_path
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

assert missing == [], missing

master = pd.read_csv(master_path)
fold_assignments = pd.read_csv(
    fold_assignment_path
)
baseline_oof = pd.read_csv(
    baseline_oof_path
)
continuation = pd.read_csv(
    continuation_path
)

print("Master:", master.shape)
print("Fold assignments:", fold_assignments.shape)
print("Baseline OOF:", baseline_oof.shape)

display(continuation)


Master: (1012, 23)
Fold assignments: (12, 5)
Baseline OOF: (24, 9)


,target,supervised_baseline_available,semi_supervised_experiment_allowed,reason
0,H1_violation,False,False,Insufficient observed class/group support in L.
1,H4_violation,False,False,Insufficient observed class/group support in L.
2,H5_violation,False,False,Insufficient observed class/group support in L.
3,H6_violation,True,True,Grouped supervised baseline completed.


## 3. Resolver targets habilitados

In [3]:
if (
    continuation[
        "semi_supervised_experiment_allowed"
    ].dtype
    !=
    bool
):
    continuation[
        "semi_supervised_experiment_allowed"
    ] = (
        continuation[
            "semi_supervised_experiment_allowed"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )

allowed_targets = (
    continuation.loc[
        continuation[
            "semi_supervised_experiment_allowed"
        ],
        "target"
    ]
    .astype(str)
    .tolist()
)

blocked_targets = [
    target
    for target in ALL_TARGETS
    if target not in allowed_targets
]

print("SSL ALLOWED TARGETS:", allowed_targets)
print("SSL BLOCKED TARGETS:", blocked_targets)

assert len(allowed_targets) > 0

if allowed_targets != ["H6_violation"]:
    warnings.warn(
        "Los targets habilitados cambiaron. "
        "Se procesarán los definidos por el notebook 28."
    )


SSL ALLOWED TARGETS: ['H6_violation']
SSL BLOCKED TARGETS: ['H1_violation', 'H4_violation', 'H5_violation']


## 4. Construir U por target

In [4]:
u_pools = {}
u_summary_records = []

for target in allowed_targets:

    u = master[
        master[target].isna()
    ].copy()

    for feature in FEATURES:
        u[feature] = pd.to_numeric(
            u[feature],
            errors="coerce"
        )

    u = u[
        u[FEATURES]
        .notna()
        .all(axis=1)
        &
        u["master_instance_id"].notna()
    ].copy()

    u_pools[target] = u

    u_summary_records.append({
        "target":
            target,

        "U_rows":
            len(u),

        "U_groups":
            u["group_id"].nunique()
            if "group_id" in u.columns
            else np.nan,

        "source_datasets":
            " | ".join(
                sorted(
                    u["source_dataset"]
                    .dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )
            ),

        "feature_domains":
            " | ".join(
                sorted(
                    u["feature_domain"]
                    .dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )
            )
    })

u_summary = pd.DataFrame(
    u_summary_records
)

display(u_summary)


,target,U_rows,U_groups,source_datasets,feature_domains
0,H6_violation,998,998,UICrit_RICO,ANDROID_UI


## 5. Configuraciones experimentales

La configuración primaria se fija antes de observar resultados:

`PRIMARY_OVERLAP_095_CAP1`

- U debe caer dentro del min–max de cada X en `L_train`;
- confianza ≥ 0.95;
- máximo acumulado de pseudo-etiquetas por clase igual al número de ejemplos observados de esa clase en `L_train`;
- máximo 3 rondas.

Las demás configuraciones son análisis de sensibilidad.


In [5]:
EXPERIMENT_CONFIGS = [
    {
        "config_id":
            "PRIMARY_OVERLAP_095_CAP1",
        "selection_role":
            "PRIMARY",
        "pool_mode":
            "RANGE_OVERLAP",
        "confidence_threshold":
            0.95,
        "cap_multiplier":
            1.0,
        "max_rounds":
            3
    },
    {
        "config_id":
            "SENS_OVERLAP_090_CAP1",
        "selection_role":
            "SENSITIVITY",
        "pool_mode":
            "RANGE_OVERLAP",
        "confidence_threshold":
            0.90,
        "cap_multiplier":
            1.0,
        "max_rounds":
            3
    },
    {
        "config_id":
            "SENS_OVERLAP_099_CAP1",
        "selection_role":
            "SENSITIVITY",
        "pool_mode":
            "RANGE_OVERLAP",
        "confidence_threshold":
            0.99,
        "cap_multiplier":
            1.0,
        "max_rounds":
            3
    },
    {
        "config_id":
            "SENS_ALLU_095_CAP1",
        "selection_role":
            "SENSITIVITY",
        "pool_mode":
            "ALL_U",
        "confidence_threshold":
            0.95,
        "cap_multiplier":
            1.0,
        "max_rounds":
            3
    },
    {
        "config_id":
            "SENS_OVERLAP_095_CAP2",
        "selection_role":
            "SENSITIVITY",
        "pool_mode":
            "RANGE_OVERLAP",
        "confidence_threshold":
            0.95,
        "cap_multiplier":
            2.0,
        "max_rounds":
            3
    }
]

experiment_config = pd.DataFrame(
    EXPERIMENT_CONFIGS
)

display(experiment_config)


,config_id,selection_role,pool_mode,confidence_threshold,cap_multiplier,max_rounds
0,PRIMARY_OVERLAP_095_CAP1,PRIMARY,RANGE_OVERLAP,0.95,1.0,3
1,SENS_OVERLAP_090_CAP1,SENSITIVITY,RANGE_OVERLAP,0.90,1.0,3
2,SENS_OVERLAP_099_CAP1,SENSITIVITY,RANGE_OVERLAP,0.99,1.0,3
3,SENS_ALLU_095_CAP1,SENSITIVITY,ALL_U,0.95,1.0,3
4,SENS_OVERLAP_095_CAP2,SENSITIVITY,RANGE_OVERLAP,0.95,2.0,3


## 6. Modelo y métricas

In [6]:
def make_rf():

    return RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


def safe_roc_auc(
    y_true,
    y_prob
):

    if np.unique(y_true).size < 2:
        return np.nan

    return roc_auc_score(
        y_true,
        y_prob
    )


def compute_metrics(
    y_true,
    y_pred,
    y_prob
):

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "n":
            len(y_true),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "precision_1":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall_1":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "f1_1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                y_pred
            ),

        "roc_auc":
            safe_roc_auc(
                y_true,
                y_prob
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                y_prob
            ),

        "brier":
            brier_score_loss(
                y_true,
                y_prob
            ),

        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


## 7. Filtro de solapamiento de dominio

In [7]:
def filter_u_pool(
    U,
    X_train,
    mode
):

    bounds_records = []

    for feature in FEATURES:
        bounds_records.append({
            "feature":
                feature,

            "train_min":
                float(
                    X_train[feature].min()
                ),

            "train_max":
                float(
                    X_train[feature].max()
                )
        })

    bounds = pd.DataFrame(
        bounds_records
    )

    if mode == "ALL_U":
        return U.copy(), bounds

    if mode != "RANGE_OVERLAP":
        raise ValueError(
            f"Pool mode desconocido: {mode}"
        )

    mask = pd.Series(
        True,
        index=U.index
    )

    for feature in FEATURES:

        lower = float(
            X_train[feature].min()
        )

        upper = float(
            X_train[feature].max()
        )

        mask &= (
            U[feature]
            >=
            lower
        )

        mask &= (
            U[feature]
            <=
            upper
        )

    return U[mask].copy(), bounds


## 8. Self-training controlado

In [8]:

def controlled_self_training(
    target,
    config,
    L_train,
    L_test,
    U_pool,
    fold_id
):

    X_train_observed = (
        L_train[FEATURES]
        .astype(float)
        .reset_index(drop=True)
    )

    y_train_observed = (
        L_train[target]
        .astype(int)
        .reset_index(drop=True)
    )

    X_test = L_test[
        FEATURES
    ].astype(float)

    y_test = (
        L_test[target]
        .astype(int)
        .to_numpy()
    )

    U_eligible, bounds = filter_u_pool(
        U_pool.copy(),
        X_train_observed,
        config["pool_mode"]
    )

    observed_counts = {
        0:
            int(
                (
                    y_train_observed
                    ==
                    0
                ).sum()
            ),

        1:
            int(
                (
                    y_train_observed
                    ==
                    1
                ).sum()
            )
    }

    total_caps = {
        cls:
            int(
                np.floor(
                    observed_counts[cls]
                    *
                    float(
                        config[
                            "cap_multiplier"
                        ]
                    )
                )
            )
        for cls in [0, 1]
    }

    selected_ids = set()
    pseudo_rows = []
    pseudo_X_parts = []
    pseudo_y_parts = []
    round_records = []

    for round_id in range(
        1,
        int(
            config[
                "max_rounds"
            ]
        )
        +
        1
    ):

        X_fit = pd.concat(
            [X_train_observed]
            +
            pseudo_X_parts,
            ignore_index=True
        )

        y_fit = pd.concat(
            [y_train_observed]
            +
            pseudo_y_parts,
            ignore_index=True
        )

        model = make_rf()

        model.fit(
            X_fit,
            y_fit
        )

        remaining = U_eligible[
            ~U_eligible[
                "master_instance_id"
            ].isin(
                selected_ids
            )
        ].copy()

        if remaining.empty:

            round_records.append({
                "target":
                    target,
                "config_id":
                    config["config_id"],
                "fold_id":
                    fold_id,
                "round_id":
                    round_id,
                "eligible_pool_initial":
                    len(U_eligible),
                "remaining_before_round":
                    0,
                "above_threshold":
                    0,
                "selected_0":
                    0,
                "selected_1":
                    0,
                "selected_total":
                    0,
                "cumulative_selected":
                    len(selected_ids)
            })

            break

        class_index = list(
            model.classes_
        ).index(1)

        prob = model.predict_proba(
            remaining[FEATURES]
            .astype(float)
        )[:, class_index]

        remaining[
            "__prob_1"
        ] = prob

        remaining[
            "__pseudo_y"
        ] = (
            remaining[
                "__prob_1"
            ]
            >=
            PREDICTION_THRESHOLD
        ).astype(int)

        remaining[
            "__confidence"
        ] = np.maximum(
            remaining[
                "__prob_1"
            ],
            1.0
            -
            remaining[
                "__prob_1"
            ]
        )

        confident = remaining[
            remaining[
                "__confidence"
            ]
            >=
            float(
                config[
                    "confidence_threshold"
                ]
            )
        ].copy()

        already_counts = {
            cls:
                sum(
                    int(
                        row[
                            "pseudo_y"
                        ]
                    )
                    ==
                    cls
                    for row in pseudo_rows
                )
            for cls in [0, 1]
        }

        selected_frames = []

        for class_value in [0, 1]:

            remaining_cap = (
                total_caps[
                    class_value
                ]
                -
                already_counts[
                    class_value
                ]
            )

            if remaining_cap <= 0:
                continue

            class_candidates = confident[
                confident[
                    "__pseudo_y"
                ]
                ==
                class_value
            ].sort_values(
                [
                    "__confidence",
                    "master_instance_id"
                ],
                ascending=[
                    False,
                    True
                ]
            ).head(
                remaining_cap
            )

            if not class_candidates.empty:
                selected_frames.append(
                    class_candidates
                )

        if selected_frames:

            round_df = pd.concat(
                selected_frames,
                ignore_index=False
            )

        else:

            round_df = pd.DataFrame(
                columns=remaining.columns
            )

        if round_df.empty:

            round_records.append({
                "target":
                    target,
                "config_id":
                    config["config_id"],
                "fold_id":
                    fold_id,
                "round_id":
                    round_id,
                "eligible_pool_initial":
                    len(U_eligible),
                "remaining_before_round":
                    len(remaining),
                "above_threshold":
                    len(confident),
                "selected_0":
                    0,
                "selected_1":
                    0,
                "selected_total":
                    0,
                "cumulative_selected":
                    len(selected_ids)
            })

            break

        round_X = (
            round_df[FEATURES]
            .astype(float)
            .reset_index(drop=True)
        )

        round_y = (
            round_df[
                "__pseudo_y"
            ]
            .astype(int)
            .reset_index(drop=True)
        )

        pseudo_X_parts.append(
            round_X
        )

        pseudo_y_parts.append(
            round_y
        )

        for _, row in (
            round_df.iterrows()
        ):

            selected_ids.add(
                row[
                    "master_instance_id"
                ]
            )

            pseudo_rows.append({
                "target":
                    target,
                "config_id":
                    config["config_id"],
                "selection_role":
                    config[
                        "selection_role"
                    ],
                "fold_id":
                    fold_id,
                "round_id":
                    round_id,
                "master_instance_id":
                    row[
                        "master_instance_id"
                    ],
                "group_id":
                    row.get(
                        "group_id",
                        ""
                    ),
                "source_dataset":
                    row.get(
                        "source_dataset",
                        ""
                    ),
                "feature_domain":
                    row.get(
                        "feature_domain",
                        ""
                    ),
                "pseudo_y":
                    int(
                        row[
                            "__pseudo_y"
                        ]
                    ),
                "prob_1":
                    float(
                        row[
                            "__prob_1"
                        ]
                    ),
                "confidence":
                    float(
                        row[
                            "__confidence"
                        ]
                    ),
                "confidence_threshold":
                    float(
                        config[
                            "confidence_threshold"
                        ]
                    ),
                "pool_mode":
                    config[
                        "pool_mode"
                    ],
                "cap_multiplier":
                    float(
                        config[
                            "cap_multiplier"
                        ]
                    )
            })

        round_records.append({
            "target":
                target,
            "config_id":
                config["config_id"],
            "fold_id":
                fold_id,
            "round_id":
                round_id,
            "eligible_pool_initial":
                len(U_eligible),
            "remaining_before_round":
                len(remaining),
            "above_threshold":
                len(confident),
            "selected_0":
                int(
                    (
                        round_y
                        ==
                        0
                    ).sum()
                ),
            "selected_1":
                int(
                    (
                        round_y
                        ==
                        1
                    ).sum()
                ),
            "selected_total":
                len(round_y),
            "cumulative_selected":
                len(selected_ids)
        })

    X_final = pd.concat(
        [X_train_observed]
        +
        pseudo_X_parts,
        ignore_index=True
    )

    y_final = pd.concat(
        [y_train_observed]
        +
        pseudo_y_parts,
        ignore_index=True
    )

    final_model = make_rf()

    final_model.fit(
        X_final,
        y_final
    )

    class_index = list(
        final_model.classes_
    ).index(1)

    test_prob = final_model.predict_proba(
        X_test
    )[:, class_index]

    test_pred = (
        test_prob
        >=
        PREDICTION_THRESHOLD
    ).astype(int)

    return {
        "metrics":
            compute_metrics(
                y_test,
                test_pred,
                test_prob
            ),

        "test_prob":
            test_prob,

        "test_pred":
            test_pred,

        "pseudo_rows":
            pseudo_rows,

        "round_records":
            round_records,

        "bounds":
            bounds,

        "eligible_U_rows":
            len(U_eligible),

        "selected_pseudo_rows":
            len(pseudo_rows),

        "final_training_rows":
            len(y_final)
    }


## 9. Ejecutar sobre los folds congelados

In [9]:
ssl_oof_records = []
ssl_fold_metric_records = []
pseudo_label_records = []
round_audit_records = []
domain_bound_records = []
test_isolation_records = []

for target in allowed_targets:

    target_assignments = (
        fold_assignments[
            fold_assignments[
                "target"
            ]
            ==
            target
        ]
        .copy()
    )

    L = master[
        master[
            "master_instance_id"
        ].isin(
            target_assignments[
                "master_instance_id"
            ]
        )
    ].copy()

    L = L.merge(
        target_assignments[
            [
                "master_instance_id",
                "evaluation_fold"
            ]
        ],
        on="master_instance_id",
        how="inner",
        validate="one_to_one"
    )

    for feature in FEATURES:
        L[feature] = pd.to_numeric(
            L[feature],
            errors="coerce"
        )

    L[target] = pd.to_numeric(
        L[target],
        errors="coerce"
    )

    U = u_pools[
        target
    ].copy()

    fold_ids = sorted(
        L[
            "evaluation_fold"
        ]
        .unique()
        .tolist()
    )

    for config in EXPERIMENT_CONFIGS:

        for fold_id in fold_ids:

            L_test = L[
                L[
                    "evaluation_fold"
                ]
                ==
                fold_id
            ].copy()

            L_train = L[
                L[
                    "evaluation_fold"
                ]
                !=
                fold_id
            ].copy()

            test_ids = set(
                L_test[
                    "master_instance_id"
                ]
            )

            train_ids = set(
                L_train[
                    "master_instance_id"
                ]
            )

            U_ids = set(
                U[
                    "master_instance_id"
                ]
            )

            isolation_ok = (
                len(
                    test_ids.intersection(
                        train_ids
                    )
                )
                ==
                0
                and
                len(
                    test_ids.intersection(
                        U_ids
                    )
                )
                ==
                0
            )

            test_isolation_records.append({
                "target":
                    target,
                "config_id":
                    config[
                        "config_id"
                    ],
                "fold_id":
                    fold_id,
                "isolation_ok":
                    isolation_ok
            })

            assert isolation_ok

            result = controlled_self_training(
                target=target,
                config=config,
                L_train=L_train,
                L_test=L_test,
                U_pool=U,
                fold_id=fold_id
            )

            for _, bound in (
                result[
                    "bounds"
                ].iterrows()
            ):

                domain_bound_records.append({
                    "target":
                        target,
                    "config_id":
                        config[
                            "config_id"
                        ],
                    "fold_id":
                        fold_id,
                    "feature":
                        bound[
                            "feature"
                        ],
                    "train_min":
                        bound[
                            "train_min"
                        ],
                    "train_max":
                        bound[
                            "train_max"
                        ],
                    "eligible_U_rows":
                        result[
                            "eligible_U_rows"
                        ]
                })

            pseudo_label_records.extend(
                result[
                    "pseudo_rows"
                ]
            )

            round_audit_records.extend(
                result[
                    "round_records"
                ]
            )

            ssl_fold_metric_records.append({
                "target":
                    target,
                "config_id":
                    config[
                        "config_id"
                    ],
                "selection_role":
                    config[
                        "selection_role"
                    ],
                "fold_id":
                    fold_id,
                "observed_train_rows":
                    len(
                        L_train
                    ),
                "test_rows":
                    len(
                        L_test
                    ),
                "eligible_U_rows":
                    result[
                        "eligible_U_rows"
                    ],
                "selected_pseudo_rows":
                    result[
                        "selected_pseudo_rows"
                    ],
                "final_training_rows":
                    result[
                        "final_training_rows"
                    ],
                **result[
                    "metrics"
                ]
            })

            L_test_reset = (
                L_test
                .reset_index(
                    drop=True
                )
            )

            for idx in range(
                len(
                    L_test_reset
                )
            ):

                ssl_oof_records.append({
                    "target":
                        target,
                    "config_id":
                        config[
                            "config_id"
                        ],
                    "selection_role":
                        config[
                            "selection_role"
                        ],
                    "master_instance_id":
                        L_test_reset.loc[
                            idx,
                            "master_instance_id"
                        ],
                    "group_id":
                        L_test_reset.loc[
                            idx,
                            "group_id"
                        ],
                    "evaluation_fold":
                        int(
                            fold_id
                        ),
                    "y_true":
                        int(
                            L_test_reset.loc[
                                idx,
                                target
                            ]
                        ),
                    "y_prob":
                        float(
                            result[
                                "test_prob"
                            ][
                                idx
                            ]
                        ),
                    "y_pred":
                        int(
                            result[
                                "test_pred"
                            ][
                                idx
                            ]
                        )
                })

ssl_oof_predictions = pd.DataFrame(
    ssl_oof_records
)

ssl_fold_metrics = pd.DataFrame(
    ssl_fold_metric_records
)

pseudo_labels = pd.DataFrame(
    pseudo_label_records
)

round_audit = pd.DataFrame(
    round_audit_records
)

domain_bounds = pd.DataFrame(
    domain_bound_records
)

test_isolation_audit = pd.DataFrame(
    test_isolation_records
)

display(ssl_fold_metrics)
display(pseudo_labels.head(30))
display(round_audit)


,target,config_id,selection_role,fold_id,observed_train_rows,test_rows,eligible_U_rows,selected_pseudo_rows,final_training_rows,n,accuracy,balanced_accuracy,precision_1,recall_1,f1_1,mcc,roc_auc,pr_auc,brier,tn,fp,fn,tp
0,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,0,8,4,141,6,14,4,0.75,0.5,0.0,0.0,0.0,0.0,1.000000,1.00,0.160209,3,0,1,0
1,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,1,8,4,202,0,8,4,0.75,0.5,0.0,0.0,0.0,0.0,0.000000,0.25,0.273806,3,0,1,0
2,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,2,8,4,117,2,10,4,0.75,0.5,0.0,0.0,0.0,0.0,1.000000,1.00,0.160575,3,0,1,0
3,H6_violation,SENS_OVERLAP_090_CAP1,SENSITIVITY,0,8,4,141,6,14,4,0.75,0.5,0.0,0.0,0.0,0.0,1.000000,1.00,0.160209,3,0,1,0
4,H6_violation,SENS_OVERLAP_090_CAP1,SENSITIVITY,1,8,4,202,0,8,4,0.75,0.5,0.0,0.0,0.0,0.0,0.000000,0.25,0.273806,3,0,1,0
5,H6_violation,SENS_OVERLAP_090_CAP1,SENSITIVITY,2,8,4,117,6,14,4,0.75,0.5,0.0,0.0,0.0,0.0,1.000000,1.00,0.153478,3,0,1,0
6,H6_violation,SENS_OVERLAP_099_CAP1,SENSITIVITY,0,8,4,141,0,8,4,1.00,1.0,1.0,1.0,1.0,1.0,1.000000,1.00,0.166780,3,0,0,1
7,H6_violation,SENS_OVERLAP_099_CAP1,SENSITIVITY,1,8,4,202,0,8,4,0.75,0.5,0.0,0.0,0.0,0.0,0.000000,0.25,0.273806,3,0,1,0
8,H6_violation,SENS_OVERLAP_099_CAP1,SENSITIVITY,2,8,4,117,0,8,4,0.75,0.5,0.0,0.0,0.0,0.0,0.666667,0.50,0.169290,3,0,1,0
9,H6_violation,SENS_ALLU_095_CAP1,SENSITIVITY,0,8,4,998,6,14,4,0.75,0.5,0.0,0.0,0.0,0.0,1.000000,1.00,0.160209,3,0,1,0


,target,config_id,selection_role,fold_id,round_id,master_instance_id,group_id,source_dataset,feature_domain,pseudo_y,prob_1,confidence,confidence_threshold,pool_mode,cap_multiplier
0,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,0,1,uicrit_rico__7025,rico__7025,UICrit_RICO,ANDROID_UI,0,0.038,0.962,0.95,RANGE_OVERLAP,1.0
1,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,0,1,uicrit_rico__30275,rico__30275,UICrit_RICO,ANDROID_UI,0,0.050,0.950,0.95,RANGE_OVERLAP,1.0
2,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,0,1,uicrit_rico__31742,rico__31742,UICrit_RICO,ANDROID_UI,0,0.050,0.950,0.95,RANGE_OVERLAP,1.0
3,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,0,1,uicrit_rico__33134,rico__33134,UICrit_RICO,ANDROID_UI,0,0.050,0.950,0.95,RANGE_OVERLAP,1.0
4,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,0,1,uicrit_rico__44015,rico__44015,UICrit_RICO,ANDROID_UI,0,0.050,0.950,0.95,RANGE_OVERLAP,1.0
5,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,0,1,uicrit_rico__56791,rico__56791,UICrit_RICO,ANDROID_UI,0,0.050,0.950,0.95,RANGE_OVERLAP,1.0
6,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,2,1,uicrit_rico__7025,rico__7025,UICrit_RICO,ANDROID_UI,0,0.044,0.956,0.95,RANGE_OVERLAP,1.0
7,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,2,2,uicrit_rico__54468,rico__54468,UICrit_RICO,ANDROID_UI,0,0.042,0.958,0.95,RANGE_OVERLAP,1.0
8,H6_violation,SENS_OVERLAP_090_CAP1,SENSITIVITY,0,1,uicrit_rico__7025,rico__7025,UICrit_RICO,ANDROID_UI,0,0.038,0.962,0.90,RANGE_OVERLAP,1.0
9,H6_violation,SENS_OVERLAP_090_CAP1,SENSITIVITY,0,1,uicrit_rico__30275,rico__30275,UICrit_RICO,ANDROID_UI,0,0.050,0.950,0.90,RANGE_OVERLAP,1.0


,target,config_id,fold_id,round_id,eligible_pool_initial,remaining_before_round,above_threshold,selected_0,selected_1,selected_total,cumulative_selected
0,H6_violation,PRIMARY_OVERLAP_095_CAP1,0,1,141,141,6,6,0,6,6
1,H6_violation,PRIMARY_OVERLAP_095_CAP1,0,2,141,135,14,0,0,0,6
2,H6_violation,PRIMARY_OVERLAP_095_CAP1,1,1,202,202,0,0,0,0,0
3,H6_violation,PRIMARY_OVERLAP_095_CAP1,2,1,117,117,1,1,0,1,1
4,H6_violation,PRIMARY_OVERLAP_095_CAP1,2,2,117,116,1,1,0,1,2
5,H6_violation,PRIMARY_OVERLAP_095_CAP1,2,3,117,115,0,0,0,0,2
6,H6_violation,SENS_OVERLAP_090_CAP1,0,1,141,141,20,6,0,6,6
7,H6_violation,SENS_OVERLAP_090_CAP1,0,2,141,135,28,0,0,0,6
8,H6_violation,SENS_OVERLAP_090_CAP1,1,1,202,202,0,0,0,0,0
9,H6_violation,SENS_OVERLAP_090_CAP1,2,1,117,117,2,2,0,2,2


## 10. Métricas OOF agregadas

In [10]:
aggregate_records = []

for (
    target,
    config_id
), group in (
    ssl_oof_predictions
    .groupby(
        [
            "target",
            "config_id"
        ]
    )
):

    config_row = (
        experiment_config[
            experiment_config[
                "config_id"
            ]
            ==
            config_id
        ]
        .iloc[0]
    )

    aggregate_records.append({
        "target":
            target,
        "config_id":
            config_id,
        "selection_role":
            config_row[
                "selection_role"
            ],
        "pool_mode":
            config_row[
                "pool_mode"
            ],
        "confidence_threshold":
            config_row[
                "confidence_threshold"
            ],
        "cap_multiplier":
            config_row[
                "cap_multiplier"
            ],
        **compute_metrics(
            group[
                "y_true"
            ].to_numpy(),
            group[
                "y_pred"
            ].to_numpy(),
            group[
                "y_prob"
            ].to_numpy()
        )
    })

ssl_aggregate_metrics = pd.DataFrame(
    aggregate_records
)

display(ssl_aggregate_metrics)


,target,config_id,selection_role,pool_mode,confidence_threshold,cap_multiplier,n,accuracy,balanced_accuracy,precision_1,recall_1,f1_1,mcc,roc_auc,pr_auc,brier,tn,fp,fn,tp
0,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,RANGE_OVERLAP,0.95,1.0,12,0.750000,0.500000,0.0,0.000000,0.0,0.000000,0.555556,0.535354,0.198197,9,0,3,0
1,H6_violation,SENS_ALLU_095_CAP1,SENSITIVITY,ALL_U,0.95,1.0,12,0.750000,0.500000,0.0,0.000000,0.0,0.000000,0.555556,0.550000,0.199046,9,0,3,0
2,H6_violation,SENS_OVERLAP_090_CAP1,SENSITIVITY,RANGE_OVERLAP,0.90,1.0,12,0.750000,0.500000,0.0,0.000000,0.0,0.000000,0.592593,0.544444,0.195831,9,0,3,0
3,H6_violation,SENS_OVERLAP_095_CAP2,SENSITIVITY,RANGE_OVERLAP,0.95,2.0,12,0.750000,0.500000,0.0,0.000000,0.0,0.000000,0.518519,0.368687,0.205536,9,0,3,0
4,H6_violation,SENS_OVERLAP_099_CAP1,SENSITIVITY,RANGE_OVERLAP,0.99,1.0,12,0.833333,0.666667,1.0,0.333333,0.5,0.522233,0.481481,0.511905,0.203292,9,0,2,1


## 11. Resumen del pseudo-etiquetado

In [11]:
pseudo_summary_records = []

for target in allowed_targets:

    U_total = len(
        u_pools[target]
    )

    for config in EXPERIMENT_CONFIGS:

        config_id = config[
            "config_id"
        ]

        if pseudo_labels.empty:

            config_pseudo = pd.DataFrame()

        else:

            config_pseudo = pseudo_labels[
                (
                    pseudo_labels[
                        "target"
                    ]
                    ==
                    target
                )
                &
                (
                    pseudo_labels[
                        "config_id"
                    ]
                    ==
                    config_id
                )
            ]

        fold_rows = ssl_fold_metrics[
            (
                ssl_fold_metrics[
                    "target"
                ]
                ==
                target
            )
            &
            (
                ssl_fold_metrics[
                    "config_id"
                ]
                ==
                config_id
            )
        ]

        pseudo_summary_records.append({
            "target":
                target,
            "config_id":
                config_id,
            "selection_role":
                config[
                    "selection_role"
                ],
            "U_total":
                U_total,
            "mean_eligible_U_per_fold":
                float(
                    fold_rows[
                        "eligible_U_rows"
                    ].mean()
                )
                if not fold_rows.empty
                else np.nan,
            "mean_selected_pseudo_per_fold":
                float(
                    fold_rows[
                        "selected_pseudo_rows"
                    ].mean()
                )
                if not fold_rows.empty
                else np.nan,
            "total_pseudo_selection_events":
                len(
                    config_pseudo
                ),
            "pseudo_0_events":
                int(
                    (
                        config_pseudo[
                            "pseudo_y"
                        ]
                        ==
                        0
                    ).sum()
                )
                if not config_pseudo.empty
                else 0,
            "pseudo_1_events":
                int(
                    (
                        config_pseudo[
                            "pseudo_y"
                        ]
                        ==
                        1
                    ).sum()
                )
                if not config_pseudo.empty
                else 0
        })

pseudo_summary = pd.DataFrame(
    pseudo_summary_records
)

display(pseudo_summary)


,target,config_id,selection_role,U_total,mean_eligible_U_per_fold,mean_selected_pseudo_per_fold,total_pseudo_selection_events,pseudo_0_events,pseudo_1_events
0,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,998,153.333333,2.666667,8,8,0
1,H6_violation,SENS_OVERLAP_090_CAP1,SENSITIVITY,998,153.333333,4.000000,12,12,0
2,H6_violation,SENS_OVERLAP_099_CAP1,SENSITIVITY,998,153.333333,0.000000,0,0,0
3,H6_violation,SENS_ALLU_095_CAP1,SENSITIVITY,998,998.000000,3.666667,11,11,0
4,H6_violation,SENS_OVERLAP_095_CAP2,SENSITIVITY,998,153.333333,4.666667,14,14,0


## 12. Comparación contra baseline supervisado

In [12]:
baseline_rf_oof = baseline_oof[
    baseline_oof[
        "model"
    ]
    ==
    "RANDOM_FOREST"
].copy()

comparison_records = []

for _, ssl_row in (
    ssl_aggregate_metrics.iterrows()
):

    target = ssl_row[
        "target"
    ]

    baseline_group = baseline_rf_oof[
        baseline_rf_oof[
            "target"
        ]
        ==
        target
    ]

    assert len(
        baseline_group
    ) > 0

    baseline_values = compute_metrics(
        baseline_group[
            "y_true"
        ].to_numpy(),
        baseline_group[
            "y_pred"
        ].to_numpy(),
        baseline_group[
            "y_prob"
        ].to_numpy()
    )

    comparison_records.append({
        "target":
            target,
        "config_id":
            ssl_row[
                "config_id"
            ],
        "selection_role":
            ssl_row[
                "selection_role"
            ],
        "baseline_balanced_accuracy":
            baseline_values[
                "balanced_accuracy"
            ],
        "ssl_balanced_accuracy":
            ssl_row[
                "balanced_accuracy"
            ],
        "delta_balanced_accuracy":
            ssl_row[
                "balanced_accuracy"
            ]
            -
            baseline_values[
                "balanced_accuracy"
            ],
        "baseline_f1":
            baseline_values[
                "f1_1"
            ],
        "ssl_f1":
            ssl_row[
                "f1_1"
            ],
        "delta_f1":
            ssl_row[
                "f1_1"
            ]
            -
            baseline_values[
                "f1_1"
            ],
        "baseline_mcc":
            baseline_values[
                "mcc"
            ],
        "ssl_mcc":
            ssl_row[
                "mcc"
            ],
        "delta_mcc":
            ssl_row[
                "mcc"
            ]
            -
            baseline_values[
                "mcc"
            ],
        "baseline_roc_auc":
            baseline_values[
                "roc_auc"
            ],
        "ssl_roc_auc":
            ssl_row[
                "roc_auc"
            ],
        "delta_roc_auc":
            ssl_row[
                "roc_auc"
            ]
            -
            baseline_values[
                "roc_auc"
            ],
        "baseline_pr_auc":
            baseline_values[
                "pr_auc"
            ],
        "ssl_pr_auc":
            ssl_row[
                "pr_auc"
            ],
        "delta_pr_auc":
            ssl_row[
                "pr_auc"
            ]
            -
            baseline_values[
                "pr_auc"
            ],
        "baseline_brier":
            baseline_values[
                "brier"
            ],
        "ssl_brier":
            ssl_row[
                "brier"
            ],
        "delta_brier":
            ssl_row[
                "brier"
            ]
            -
            baseline_values[
                "brier"
            ]
    })

ssl_vs_baseline = pd.DataFrame(
    comparison_records
)

display(ssl_vs_baseline)


,target,config_id,selection_role,baseline_balanced_accuracy,ssl_balanced_accuracy,delta_balanced_accuracy,baseline_f1,ssl_f1,delta_f1,baseline_mcc,ssl_mcc,delta_mcc,baseline_roc_auc,ssl_roc_auc,delta_roc_auc,baseline_pr_auc,ssl_pr_auc,delta_pr_auc,baseline_brier,ssl_brier,delta_brier
0,H6_violation,PRIMARY_OVERLAP_095_CAP1,PRIMARY,0.666667,0.500000,-0.166667,0.5,0.0,-0.5,0.522233,0.000000,-0.522233,0.481481,0.555556,0.074074,0.511905,0.535354,0.023449,0.203292,0.198197,-5.095333e-03
1,H6_violation,SENS_ALLU_095_CAP1,SENSITIVITY,0.666667,0.500000,-0.166667,0.5,0.0,-0.5,0.522233,0.000000,-0.522233,0.481481,0.555556,0.074074,0.511905,0.550000,0.038095,0.203292,0.199046,-4.245667e-03
2,H6_violation,SENS_OVERLAP_090_CAP1,SENSITIVITY,0.666667,0.500000,-0.166667,0.5,0.0,-0.5,0.522233,0.000000,-0.522233,0.481481,0.592593,0.111111,0.511905,0.544444,0.032540,0.203292,0.195831,-7.461000e-03
3,H6_violation,SENS_OVERLAP_095_CAP2,SENSITIVITY,0.666667,0.500000,-0.166667,0.5,0.0,-0.5,0.522233,0.000000,-0.522233,0.481481,0.518519,0.037037,0.511905,0.368687,-0.143218,0.203292,0.205536,2.244000e-03
4,H6_violation,SENS_OVERLAP_099_CAP1,SENSITIVITY,0.666667,0.666667,0.000000,0.5,0.5,0.0,0.522233,0.522233,0.000000,0.481481,0.481481,0.000000,0.511905,0.511905,0.000000,0.203292,0.203292,-2.775558e-17


## 13. Estabilidad de pseudo-etiquetas

In [13]:
stability_records = []

if not pseudo_labels.empty:

    for (
        target,
        config_id,
        instance_id
    ), group in (
        pseudo_labels
        .groupby(
            [
                "target",
                "config_id",
                "master_instance_id"
            ]
        )
    ):

        dominant_share = (
            group[
                "pseudo_y"
            ]
            .astype(int)
            .value_counts(
                normalize=True
            )
            .max()
        )

        stability_records.append({
            "target":
                target,
            "config_id":
                config_id,
            "master_instance_id":
                instance_id,
            "selection_events":
                len(
                    group
                ),
            "folds_selected":
                group[
                    "fold_id"
                ].nunique(),
            "dominant_label_share":
                float(
                    dominant_share
                ),
            "fully_stable_when_selected":
                bool(
                    dominant_share
                    ==
                    1.0
                )
        })

pseudo_label_stability = pd.DataFrame(
    stability_records
)

display(
    pseudo_label_stability.head(30)
)


,target,config_id,master_instance_id,selection_events,folds_selected,dominant_label_share,fully_stable_when_selected
0,H6_violation,PRIMARY_OVERLAP_095_CAP1,uicrit_rico__30275,1,1,1.0,True
1,H6_violation,PRIMARY_OVERLAP_095_CAP1,uicrit_rico__31742,1,1,1.0,True
2,H6_violation,PRIMARY_OVERLAP_095_CAP1,uicrit_rico__33134,1,1,1.0,True
3,H6_violation,PRIMARY_OVERLAP_095_CAP1,uicrit_rico__44015,1,1,1.0,True
4,H6_violation,PRIMARY_OVERLAP_095_CAP1,uicrit_rico__54468,1,1,1.0,True
5,H6_violation,PRIMARY_OVERLAP_095_CAP1,uicrit_rico__56791,1,1,1.0,True
6,H6_violation,PRIMARY_OVERLAP_095_CAP1,uicrit_rico__7025,2,2,1.0,True
7,H6_violation,SENS_ALLU_095_CAP1,uicrit_rico__12597,1,1,1.0,True
8,H6_violation,SENS_ALLU_095_CAP1,uicrit_rico__30275,1,1,1.0,True
9,H6_violation,SENS_ALLU_095_CAP1,uicrit_rico__31742,1,1,1.0,True


## 14. Decisión del notebook 29

In [14]:
isolation_ok = bool(
    test_isolation_audit[
        "isolation_ok"
    ].all()
)

training_completed = bool(
    isolation_ok
    and
    len(
        ssl_aggregate_metrics
    )
    >
    0
)

primary_pseudo = pseudo_summary[
    pseudo_summary[
        "selection_role"
    ]
    ==
    "PRIMARY"
]

decision = pd.DataFrame([
    {
        "test_isolation_ok":
            isolation_ok,
        "ssl_training_completed":
            training_completed,
        "allowed_targets":
            " | ".join(
                allowed_targets
            ),
        "blocked_targets":
            " | ".join(
                blocked_targets
            ),
        "primary_config":
            "PRIMARY_OVERLAP_095_CAP1",
        "primary_pseudo_selection_events":
            int(
                primary_pseudo[
                    "total_pseudo_selection_events"
                ].sum()
            )
            if not primary_pseudo.empty
            else 0,
        "ready_for_notebook_30":
            training_completed,
        "global_status":
            (
                "READY_FOR_30_MODEL_EVALUATION"
                if training_completed
                else
                "SSL_EXECUTION_BLOCKED"
            )
    }
])

display(decision)


,test_isolation_ok,ssl_training_completed,allowed_targets,blocked_targets,primary_config,primary_pseudo_selection_events,ready_for_notebook_30,global_status
0,True,True,H6_violation,H1_violation | H4_violation | H5_violation,PRIMARY_OVERLAP_095_CAP1,8,True,READY_FOR_30_MODEL_EVALUATION


## 15. Guardar artefactos

In [15]:
experiment_config.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_experiment_config.csv",
    index=False
)

u_summary.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_unlabeled_pool_summary.csv",
    index=False
)

test_isolation_audit.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_test_isolation_audit.csv",
    index=False
)

domain_bounds.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_domain_overlap_bounds.csv",
    index=False
)

round_audit.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_round_audit.csv",
    index=False
)

pseudo_labels.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_pseudo_labels.csv",
    index=False
)

pseudo_summary.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_pseudo_label_summary.csv",
    index=False
)

pseudo_label_stability.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_pseudo_label_stability.csv",
    index=False
)

ssl_oof_predictions.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_oof_predictions.csv",
    index=False
)

ssl_fold_metrics.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_fold_metrics.csv",
    index=False
)

ssl_aggregate_metrics.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_aggregate_metrics.csv",
    index=False
)

ssl_vs_baseline.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_vs_supervised_baseline.csv",
    index=False
)

decision.to_csv(
    SEMISUP_REPORTS_DIR
    / "ssl_training_decision.csv",
    index=False
)

with open(
    SEMISUP_REPORTS_DIR
    / "ssl_training_contract.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        {
            "features":
                FEATURES,
            "allowed_targets":
                allowed_targets,
            "blocked_targets":
                blocked_targets,
            "prediction_threshold":
                PREDICTION_THRESHOLD,
            "random_state":
                RANDOM_STATE,
            "primary_config":
                "PRIMARY_OVERLAP_095_CAP1",
            "rule":
                (
                    "Pseudo-labels are training-only "
                    "and never ground truth."
                )
        },
        file,
        indent=2
    )


In [16]:
expected_outputs = [
    "ssl_experiment_config.csv",
    "ssl_unlabeled_pool_summary.csv",
    "ssl_test_isolation_audit.csv",
    "ssl_domain_overlap_bounds.csv",
    "ssl_round_audit.csv",
    "ssl_pseudo_labels.csv",
    "ssl_pseudo_label_summary.csv",
    "ssl_pseudo_label_stability.csv",
    "ssl_oof_predictions.csv",
    "ssl_fold_metrics.csv",
    "ssl_aggregate_metrics.csv",
    "ssl_vs_supervised_baseline.csv",
    "ssl_training_decision.csv",
    "ssl_training_contract.json"
]

for filename in expected_outputs:

    path = (
        SEMISUP_REPORTS_DIR
        /
        filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )


ssl_experiment_config.csv ✅
ssl_unlabeled_pool_summary.csv ✅
ssl_test_isolation_audit.csv ✅
ssl_domain_overlap_bounds.csv ✅
ssl_round_audit.csv ✅
ssl_pseudo_labels.csv ✅
ssl_pseudo_label_summary.csv ✅
ssl_pseudo_label_stability.csv ✅
ssl_oof_predictions.csv ✅
ssl_fold_metrics.csv ✅
ssl_aggregate_metrics.csv ✅
ssl_vs_supervised_baseline.csv ✅
ssl_training_decision.csv ✅
ssl_training_contract.json ✅


## Punto de detención

No seleccionar todavía un modelo final.

El notebook 30 deberá revisar conjuntamente:

```text
baseline_aggregate_metrics.csv
ssl_aggregate_metrics.csv
ssl_vs_supervised_baseline.csv
ssl_pseudo_label_summary.csv
ssl_pseudo_label_stability.csv
baseline_provenance_by_class.csv
```

La evaluación final debe considerar:

- balanced accuracy;
- F1 y MCC;
- PR-AUC y ROC-AUC;
- Brier score;
- estabilidad entre folds;
- sensibilidad a confianza/cap/filtro de dominio;
- número y clase de pseudo-etiquetas;
- cambio de dominio Web→Android;
- tamaño extremadamente pequeño de L;
- mezcla de procedencia A/B/C.

El siguiente archivo será:

`30_model_evaluation.ipynb`.
